In [1]:
from datetime import date, timedelta
import sqlite3
import pandas as pd
from pybaseball import cache, statcast
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
from api_scraper import MLB_Scrape
from tqdm.auto import tqdm

In [2]:
cache.enable()
processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

In [3]:
def download_year(y):
    #get statcast data
    start_date = date(y, 1, 1)
    year = str(y)
    end_date = date(y, 12, 31)

    
    chunks = []
    while start_date <= end_date:
        chunk_end = min(start_date + timedelta(days=30), end_date)
        print(f"Downloading {start_date} through {chunk_end}")

        chunk = statcast(
            start_date.isoformat(),
            chunk_end.isoformat()
        )

        if chunk is not None and not chunk.empty:
            chunks.append(chunk)

        start_date = chunk_end + timedelta(days=1)
    
    #get umpire data
    scraper = MLB_Scrape()
    if chunks:
        statcast_data = pd.concat(chunks, ignore_index=True)
    else:
        print("No statcast data downloaded.")


    game_list = list(statcast_data["game_pk"].unique())
    game_data = scraper.get_data(game_list_input=game_list)
    ump_rows = []
    for game in game_data:
        for off in game["liveData"]["boxscore"]["officials"]:

            if off["officialType"] == "Home Plate":

                ump_rows.append({
                    "game_pk": game["gamePk"],
                    "hp_official_name": off["official"]["fullName"],
                    "hp_official_id": off["official"]["id"],
                })

    umpire_data = pd.DataFrame(ump_rows)

    
    if not umpire_data.empty and not statcast_data.empty:
        final_data = pd.merge(statcast_data, umpire_data, left_on='game_pk', right_on='game_pk', how='left')
        
    
        output_path = processed_dir / f"statcast_{year}.parquet"
        final_data.to_parquet(output_path,index=False)

        print(f"Saved Statcast {len(statcast_data):,} rows.")
        print(f"Saved Total {len(final_data):,} rows.")
        print(f"Saved {y} to {output_path}")
    else:
        print("No data downloaded.")


In [4]:
# years = [2022,2023,2024,2025]
years = [2025]

for y in tqdm(years,desc="years"):
    download_year(y)

years:   0%|          | 0/1 [00:00<?, ?it/s]

This is a large query, it may take a moment to complete
Skipping offseason dates


0it [00:00, ?it/s]

This is a large query, it may take a moment to complete


Skipping offseason dates


0it [00:00, ?it/s]

... (23 outputs left out)

0it [00:00, ?it/s]


This May Take a While. Progress Bar shows Completion of Data Retrieval.


Processing: 100%|██████████| 2626/2626 [01:11<00:00, 36.51iteration/s]


Saved Statcast 770,795 rows.
Saved Total 770,795 rows.
Saved 2025 to ..\data\processed\statcast_2025.parquet


In [5]:
# while start_date <= end_date:
#     chunk_end = min(start_date + timedelta(days=30), end_date)
#     print(f"Downloading {start_date} through {chunk_end}")

#     chunk = statcast(
#         start_date.isoformat(),
#         chunk_end.isoformat()
#     )

#     if chunk is not None and not chunk.empty:
#         chunks.append(chunk)

#     start_date = chunk_end + timedelta(days=1)

In [6]:
# if chunks:
#     statcast_data = pd.concat(chunks, ignore_index=True)
#     statcast_data.to_parquet(
#         processed_dir / f"statcast{year}.parquet",
#         index=False
#     )
#     print(f"Saved {len(statcast_data):,} rows.")
# else:
#     print("No data downloaded.")

In [7]:
# if chunks:
#     statcast_data = pd.concat(chunks, ignore_index=True)
#     print(statcast_data.shape)
#     print(statcast_data.columns.tolist())

In [8]:
# print(statcast_data["game_pk"].notna().sum())
# print(statcast_data["game_pk"].value_counts(dropna=False).head())

In [9]:
# from api_scraper import MLB_Scrape
# scraper = MLB_Scrape()

# game_list = list(statcast_data["game_pk"].unique())
# game_data = scraper.get_data(game_list_input=game_list)

In [10]:
# ump_rows = []
# for game in game_data:
#   for off in game["liveData"]["boxscore"]["officials"]:

#     if off["officialType"] == "Home Plate":

#       ump_rows.append({
#           "game_pk": game["gamePk"],
#           "hp_official_name": off["official"]["fullName"],
#           "hp_official_id": off["official"]["id"],
#       })

# umpire_data = pd.DataFrame(ump_rows)

In [11]:
# print(umpire_data.columns.tolist())

In [12]:
# if chunks:
#     statcast_data = pd.concat(chunks, ignore_index=True)
#     final_data = pd.merge(statcast_data, umpire_data, left_on='game_pk', right_on='game_pk', how='left')
#     final_data.to_parquet(
#         processed_dir / f"statcast_{year}.parquet",
#         index=False
#     )
#     print(f"Saved Statcast {len(statcast_data):,} rows.")
#     print(f"Saved Total {len(final_data):,} rows.")
# else:
#     print("No data downloaded.")